# Datathon26 — datasets & evaluators demo

Models don't run locally at the datathon: you submit studies to an external
**reportbench** service and score its pre-computed outputs with `radharmony.datathon26`.

- **Classification** — score the `.npz` `global` embedding with a probe.
- **Segmentation** — fold the `.npz` `patches` (+ `grid`) into a dense map; a
  frozen-feature probe scores Dice/IoU against your masks.
- **Report generation** — score the returned `report.txt` vs the reference (RadEval).

Evaluators subclass the standard ones, so k-fold, bootstrap, `macro_average`, the
full RadEval suite, and `generations.csv` all still work.

> **§1** *submits* a run with the one-shot factories; **§2** *scores finished runs*
> with the results datasets.

## The workflow

```
batch prep              reportbench service         radharmony.datathon26
----------              -------------------         ---------------------
sample + anonymize  --> submit studies          --> point a dataset at the
6 CXR datasets          (embed / report task)       results dir + batch df
                                                            |
dataset_{A..F}.csv      writes per image / study:          v
(labels/reports/masks)    <study>/<model>_<stem>.npz  evaluate() -> metrics
                          <study>/<model>_report.txt
```

You supply the **batch table** (labels / reports / masks) and the **results dir**;
a dataset joins them and an evaluator scores.

## 0. Setup

In [1]:
import pandas as pd

import radharmony.datathon26  # noqa: F401  -> registers all datathon26_* keys
from radharmony.datathon26 import (
    EmbeddingResultsDataset,
    ReportResultsDataset,
    PatchSegResultsDataset,
    DatathonLinearProbeEvaluator,
    DatathonReportGenerationEvaluator,
    DatathonLinearProbeSegEvaluator,
    viewwise_embedding_path,
    perstudy_report_path,
)
from radharmony.evaluator.metrics.language import LIGHT_METRICS
from radharmony.registry import list_datasets
from radharmony.evaluator._registry import list_evaluators

print("datasets: ", sorted(k for k in list_datasets() if "datathon" in k))
print("evaluators:", sorted(k for k in list_evaluators() if "datathon" in k))

datasets:  ['datathon26_embedding', 'datathon26_report', 'datathon26_segmentation']
evaluators: ['datathon26_conv_probe_seg', 'datathon26_knn_probe', 'datathon26_linear_probe', 'datathon26_linear_probe_seg', 'datathon26_prototype_probe', 'datathon26_report_generation', 'datathon26_svm_probe', 'datathon26_upernet_seg']


## 1. One-shot: submit F → classification + segmentation

Dataset **F (SIIM-ACR-PTX)** is the worked test: 10,000 frontal CXRs, a binary
`pneumothorax` label, and — uniquely among the six — **real segmentation masks**
(`mask_path`). It has no finished run yet, so we submit it. The **one-shot
factories** stage the harmonized frame + raw images, submit the `embed` task (via
`ReportBenchClient` → `rbclient.py`), wait, and hand back a ready dataset.

Fill in your `INSTRUCTIONS.md` values and run **on the datathon server** — it submits
real jobs. Template, not "Run All"-safe.

> - **Cls + seg share one embed job:** `DatathonEmbeddingDataset` and
>   `DatathonPatchEmbeddingDataset` submit the same `embed` task (`.npz` carry both
>   `global` and `patches`) — submit once, reuse with `skip_submit=True` + same `run_name`.
> - **No reports:** SIIM ships no findings text, so report generation lives in §2b (on A).
> - **Staging:** `prepare` copies the images once into the run folder.

In [4]:
from radharmony.datathon26 import (
    ReportBenchClient,
    DatathonEmbeddingDataset,
)
import os

# Files you prepare under DATA_DIR are owned by you, but the zeus service reads
# them as a different user — so they must be world-readable or `submit` 500s.
# umask 0o022 makes the staged + prepared files o+r. One-time, in a terminal:
#   chmod o+x <DATA_DIR>   (lets the service traverse into the batch root)
os.umask(0o022)

# --- fill these in (see your team INSTRUCTIONS.md), then run on the server ---
RBCLIENT_PATH = "<DATATHON_SKILL_DIR>/rbclient.py"
DATA_DIR = "<YOUR_BATCH_ROOT>"
API_KEY = "rb_..."  # "rb_..."
MODEL = "model-a"

# Dataset F (SIIM-ACR-PTX): frontal CXRs + pneumothorax label + real masks.
F_DIR = "<YOUR_DATATHON_CXR_DIR>/"
IMAGES_DIR = f"{F_DIR}/images"  # image_path in the frame is relative to this
df_F = pd.read_csv(
    f"{F_DIR}/dataset_F.csv"
)  # .head(200)  # small demo subset (drop .head for all)

client = ReportBenchClient(
    rbclient_path=RBCLIENT_PATH, data_dir=DATA_DIR, api_key=API_KEY
)

# Classification — submit the embed task once as run_F; the .npz `global` vector
# feeds the probe. The label column (pneumothorax) is auto-detected from the frame.
# Fresh run_name avoids colliding with an earlier prepared batch (the client
# won't overwrite an existing run).
emb_ds = DatathonEmbeddingDataset(
    df_F,
    IMAGES_DIR,
    client=client,
    model=MODEL,
    run_name="run_F_demo",
    skip_submit=True,
)
DatathonLinearProbeEvaluator(dataset=emb_ds, n_folds=5).evaluate()

extracting embeddings:   0%|          | 0/157 [00:00<?, ?batch/s]

,n_train,fold,seed,bootstrap,label,auroc,auprc,f1,accuracy,balanced_accuracy,tpr,tnr,ppv,npv,mcc,threshold
0,-1,0,-1,-1,pneumothorax,0.962462,0.873003,0.792523,0.8890,0.914169,0.959276,0.869063,0.675159,0.986880,0.740536,0.429853
1,-1,1,-1,-1,pneumothorax,0.962544,0.883797,0.810127,0.9025,0.904538,0.908297,0.900778,0.731107,0.970650,0.753508,0.558429
2,-1,2,-1,-1,pneumothorax,0.950447,0.851359,0.754435,0.8685,0.893470,0.937355,0.849586,0.631250,0.980147,0.693638,0.378273
3,-1,3,-1,-1,pneumothorax,0.959578,0.860332,0.798010,0.8985,0.908486,0.926097,0.890874,0.701049,0.977591,0.744600,0.483844
4,-1,4,-1,-1,pneumothorax,0.952718,0.851073,0.761194,0.8720,0.890293,0.923077,0.857510,0.647619,0.975182,0.697245,0.479927
5,-1,0,-1,-1,macro_average,0.962462,0.873003,0.792523,0.8890,0.914169,0.959276,0.869063,0.675159,0.986880,0.740536,0.429853
6,-1,1,-1,-1,macro_average,0.962544,0.883797,0.810127,0.9025,0.904538,0.908297,0.900778,0.731107,0.970650,0.753508,0.558429
7,-1,2,-1,-1,macro_average,0.950447,0.851359,0.754435,0.8685,0.893470,0.937355,0.849586,0.631250,0.980147,0.693638,0.378273
8,-1,3,-1,-1,macro_average,0.959578,0.860332,0.798010,0.8985,0.908486,0.926097,0.890874,0.701049,0.977591,0.744600,0.483844
9,-1,4,-1,-1,macro_average,0.952718,0.851073,0.761194,0.8720,0.890293,0.923077,0.857510,0.647619,0.975182,0.697245,0.479927


**Segmentation factory — real masks.** `DatathonPatchEmbeddingDataset` reuses the
same `embed` job (`run_F` above, so `skip_submit=True`), folds each `.npz`'s
`patches` → `[dim, H, W]` via its `grid`, and pairs it with F's real `mask_path`.
One embed job feeds both classification and segmentation.

In [3]:
from radharmony.datathon26 import DatathonPatchEmbeddingDataset

# Segmentation on F's REAL masks. mask_path in dataset_F.csv is relative
# (masks_F/<study>/frontal.png); mask_dir resolves it (masks live in their own
# tree, not under the embed outputs). Reuses the embed run_F_demo submitted above
# (skip_submit=True); drop skip_submit to submit a fresh run.
seg_ds = DatathonPatchEmbeddingDataset(
    df_F,
    IMAGES_DIR,
    client=client,
    model=MODEL,
    run_name="run_F_demo",
    skip_submit=True,  # reuse the embed run from the classification cell
    mask_col="mask_path",
    mask_dir=F_DIR,  # root for the relative mask paths
    mask_size=224,  # probe output resolution; masks resized to match
)
DatathonLinearProbeSegEvaluator(
    dataset=seg_ds,
    num_classes=2,
    n_folds=5,
    device="cpu",
    autocast_dtype=None,
).evaluate().groupby("label")[["dice", "iou"]].mean().round(4)

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

,dice,iou
label,,
class_0,0.9986,0.9972
class_1,0.2117,0.1485
macro_average,0.2117,0.1485


## 2. Score finished runs — no service call

Point a **results dataset** at a finished `_reportbench_out/<model>/` and score; the
service is never touched. Two ways to get such a folder:

- **§2a** — the run **you** submitted in §1 (dataset **F**): read its `.npz` back for
  classification + segmentation.
- **§2b** — the organizer's **shared pre-computed corpus** (dataset **A**): every
  model's outputs staged read-only. A carries reports + 14 finding labels (F has
  neither), so it drives report generation and a richer multi-label cls.

The service writes, per image / per study:

```
A_patient_00018_study_00001/
    model-a_pa.npz        # per IMAGE: `global` [dim] + `patches` [n, dim] + `grid`
    model-a_lateral.npz
    model-a_report.txt    # per STUDY
```

Match granularity with the `result_path=` resolver: embeddings/patches per-image
(`viewwise_embedding_path`), reports per-study (`perstudy_report_path`).

Registry keys: `datathon26_{embedding,segmentation,report}`,
`datathon26_{linear,knn,svm,prototype}_probe`, `datathon26_linear_probe_seg`
(+ conv/upernet), `datathon26_report_generation`.

### 2a. Read back the run you submitted (F)

§1 submitted F's `embed` job; its `.npz` are on disk now. The **results datasets**
read them with no service call — `EmbeddingResultsDataset` takes the `global` vector
(classification), `PatchSegResultsDataset` folds `patches` + `grid` into a dense map
and pairs it with F's real masks (segmentation). Needs §1 to have run.
(`DATA_DIR`, `MODEL`, `df_F`, `F_DIR` come from §1.)

In [ ]:
# Classification, read-back: point EmbeddingResultsDataset at F's finished embed run
# (submitted in §1) and score the `global` vectors — no service call. One row per
# image; viewwise_embedding_path maps each to <study>/<model>_<stem>.npz.
F_OUT_DIR = (
    f"{DATA_DIR}/run_F_demo/_reportbench_out/{MODEL}"  # <study>/model-a_frontal.npz
)

f_cls_ds = EmbeddingResultsDataset(
    F_OUT_DIR,
    harmonized_df=df_F,
    result_path=viewwise_embedding_path(MODEL),
)
DatathonLinearProbeEvaluator(
    dataset=f_cls_ds, n_folds=5, device="cpu", autocast_dtype=None, num_workers=2
).evaluate().groupby("label")[["auroc", "auprc", "f1"]].mean().round(4)

In [ ]:
# Segmentation, read-back: same F run, folds each .npz's patches -> [dim, H, W] via
# its grid and pairs with F's real masks (relative mask_path + mask_dir). Nothing
# re-runs.
seg_ds = PatchSegResultsDataset(
    F_OUT_DIR,
    harmonized_df=df_F,  # real mask_path already present (relative)
    result_path=viewwise_embedding_path(MODEL),
    mask_dir=F_DIR,  # root for the relative mask paths
    mask_size=224,
)
DatathonLinearProbeSegEvaluator(
    dataset=seg_ds,
    num_classes=2,
    n_folds=5,
    device="cpu",
    autocast_dtype=None,
    num_workers=2,
).evaluate().groupby("label")[["dice", "iou"]].mean().round(4)

### 2b. The organizer's shared corpus (A) — reports + multi-label cls

A run you didn't submit: the organizer staged every model's outputs read-only. Dataset
**A** carries reference reports and 14 finding labels, so it drives report generation
(F has none) and a richer multi-label classification. `manifest.json` lists which
images were actually submitted; filter `dataset_A.csv` to those before scoring.

In [ ]:
import json

# Your team's finished run for dataset A (paths from INSTRUCTIONS.md).
RUN_DIR = (
    "<YOUR_TEAM_RUN_ROOT>/"
    "dt26_runs/dataset_A"
)
CSV_PATH = "<YOUR_DATATHON_CXR_DIR>/dataset_A.csv"
MODEL = "model-a"
OUT_DIR = f"{RUN_DIR}/_reportbench_out/{MODEL}"  # <study>/<model>_<stem>.npz + <model>_report.txt

# manifest.json lists exactly which images were submitted for this run. Accept
# both the study-keyed schema ({"studies": {"<id>": {"images": [{"path","view"}],
# ...}}}, path = bare filename) and the older flat one ({"images": [{"path":
# "<id>/<file>", ...}]}); normalise to path="<study>/<file>".
with open(f"{RUN_DIR}/manifest.json") as f:
    manifest = json.load(f)

if "studies" in manifest:
    rows = [
        {"path": f"{sid}/{img['path']}", "view": img.get("view")}
        for sid, spec in manifest["studies"].items()
        for img in spec["images"]
    ]
else:
    rows = manifest["images"]
sub = pd.DataFrame(rows)  # columns: path ("<study>/<view>.dcm"), view
print(f"{len(sub)} submitted images")
sub.head()

In [ ]:
# Filter dataset_A to the studies actually submitted (join on the manifest paths).
# Lowercase BOTH sides: manifest paths are mixed-case ("A_patient_..."), so
# lowercasing only image_path would never match and the filter returns nothing.
df_A_sub = pd.read_csv(CSV_PATH)
df_A_sub = df_A_sub[df_A_sub.image_path.str.lower().isin(sub.path.str.lower())].copy()

print(f"{len(df_A_sub)} image-rows across {df_A_sub.study_id.nunique()} studies")
df_A_sub.head()

821 image-rows across 500 studies


,patient_id,study_id,image_path,view_position,report,atelectasis,cardiomegaly,consolidation,edema,enlarged_cardiomediastinum,fracture,lung_lesion,lung_opacity,no_finding,pleural_effusion,pleural_other,pneumonia,pneumothorax,support_devices
30,A_patient_00018,A_patient_00018_study_00001,A_patient_00018_study_00001/lateral.dcm,LATERAL,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
31,A_patient_00018,A_patient_00018_study_00001,A_patient_00018_study_00001/pa.dcm,PA,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
105,A_patient_00058,A_patient_00058_study_00001,A_patient_00058_study_00001/ap.dcm,AP,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
193,A_patient_00111,A_patient_00111_study_00001,A_patient_00111_study_00001/lateral.dcm,LATERAL,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
194,A_patient_00111,A_patient_00111_study_00001,A_patient_00111_study_00001/pa.dcm,PA,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Classification — one embedding per view -> <study>/<model>_<view>.npz. df_A_sub is
# already one row per image (the right granularity). A's 14 labels -> macro-average
# across findings, unlike F's single binary label.
cls_ds = EmbeddingResultsDataset(
    OUT_DIR,
    harmonized_df=df_A_sub,
    result_path=viewwise_embedding_path(MODEL),
)
cls_results = DatathonLinearProbeEvaluator(
    dataset=cls_ds, n_folds=5, device="cpu", autocast_dtype=None, num_workers=2
).evaluate()
cls_results.groupby("label")[["auroc", "auprc", "f1"]].mean().round(4)

In [ ]:
# Report generation — one report per study -> <study>/<model>_report.txt. Reports are
# per-study, so dedupe df_A_sub to one row per study. F can't do this (no reports).
rep_ds = ReportResultsDataset(
    OUT_DIR,
    harmonized_df=df_A_sub.drop_duplicates("study_id").copy(),
    result_path=perstudy_report_path(MODEL),
)
DatathonReportGenerationEvaluator(
    dataset=rep_ds, metrics=LIGHT_METRICS, ref_section="findings", num_workers=2
).evaluate()